# 📦 Notebook 01 — Ingestão e Armazenamento

**Projeto:** Análise de Custos e Margem por Categoria — Olist E-Commerce  
**Autor:** Ariel Marquezin  
**Stack:** PySpark · PyArrow · pandas · Azure Blob Storage  

---

## 🎯 Objetivo deste notebook

Simular uma **camada de ingestão de dados** como em pipelines reais de engenharia de dados:

1. Ler os arquivos CSV brutos do Olist com **PySpark**
2. Realizar validações básicas de qualidade
3. Converter e persistir em formato **Parquet** via **PyArrow** (padrão de Data Lakes)
4. Fazer upload dos Parquet para **Azure Blob Storage** (simulando ADLS Gen2)

> **Nota sobre SAP HANA:** Em ambiente produtivo, a ingestão viria de views do SAP/HANA.  
> O bloco de conexão está documentado abaixo como referência arquitetural.

## 0. Referência: Conexão SAP HANA (produção)

```python
# ── Conexão SAP HANA via hdbcli (ambiente produtivo) ──────────────────────────
# from hdbcli import dbapi
#
# conn = dbapi.connect(
#     address  = "hana-host.empresa.com",
#     port     = 443,
#     user     = "ANALYTICS_USER",
#     password = os.environ["HANA_PASSWORD"],
#     encrypt  = True
# )
#
# query = """
#     SELECT
#         ORDER_ID, PRODUCT_ID, CATEGORY, PRICE, FREIGHT, ORDER_STATUS, ORDER_DATE
#     FROM SALES_SCHEMA.VW_ORDERS_FULL
#     WHERE ORDER_DATE >= ADD_DAYS(CURRENT_DATE, -365)
# """
# df_hana = pd.read_sql(query, conn)
# conn.close()
#
# ── Para este projeto utilizamos o dataset público Olist como proxy ────────────
```

In [ ]:
# ── Instalação de dependências (rode apenas uma vez) ──────────────────────────
# No Databricks, PySpark já está disponível nativamente.
# Execute esta célula apenas se estiver rodando em Colab ou ambiente local.

# !pip install pyspark pyarrow pandas azure-storage-blob python-dotenv -q

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType, TimestampType
)

# PyArrow
import pyarrow as pa
import pyarrow.parquet as pq

# pandas
import pandas as pd

# Azure
from azure.storage.blob import BlobServiceClient

print(f"PyArrow version : {pa.__version__}")
print(f"Pandas version  : {pd.__version__}")
print("Imports OK ✅")

In [ ]:
# ── 2. Configurações e paths ──────────────────────────────────────────────────

# Databricks: ajuste o path para dbfs:/FileStore/olist/ se necessário
RAW_PATH    = "../data/raw/"      # CSVs originais do Kaggle
PARQUET_PATH = "../data/parquet/" # destino local antes do upload

os.makedirs(PARQUET_PATH, exist_ok=True)

# Azure — carregue via variável de ambiente (nunca hardcode credenciais)
AZURE_CONN_STRING  = os.environ.get("AZURE_STORAGE_CONNECTION_STRING", "")
AZURE_CONTAINER    = "olist-datalake"

# Arquivos esperados do dataset Olist
OLIST_FILES = [
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_products_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]

print(f"RAW_PATH    : {os.path.abspath(RAW_PATH)}")
print(f"PARQUET_PATH: {os.path.abspath(PARQUET_PATH)}")

In [ ]:
# ── 3. Inicializar SparkSession ───────────────────────────────────────────────
# No Databricks, SparkSession já existe como `spark`.
# Este bloco cria uma sessão local para ambientes externos.

try:
    spark  # verifica se já existe (Databricks)
    print("SparkSession detectada (Databricks) ✅")
except NameError:
    spark = (
        SparkSession.builder
        .appName("Olist-Ingestao")
        .config("spark.sql.shuffle.partitions", "8")   # leve para dataset pequeno
        .config("spark.driver.memory", "2g")
        .getOrCreate()
    )
    print(f"SparkSession local criada ✅  |  Spark {spark.version}")

In [ ]:
# ── 4. Leitura dos CSVs com PySpark ──────────────────────────────────────────
# inferSchema=True faz uma passagem extra nos dados para inferir tipos;
# em produção, use schema explícito para performance.

def read_csv_spark(filename: str):
    path = os.path.join(RAW_PATH, filename)
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("encoding", "UTF-8")
        .csv(path)
    )
    print(f"  {filename:<55} → {df.count():>7,} linhas | {len(df.columns)} colunas")
    return df

print("Lendo arquivos CSV...\n")
dfs = {f.replace(".csv", ""): read_csv_spark(f) for f in OLIST_FILES}
print("\nLeitura concluída ✅")

In [ ]:
# ── 5. Validação de qualidade — nulls e duplicatas ────────────────────────────

def quality_report(name: str, df) -> pd.DataFrame:
    """Retorna um relatório de nulos e % de completude por coluna."""
    total = df.count()
    rows = []
    for col in df.columns:
        nulls = df.filter(F.col(col).isNull()).count()
        rows.append({
            "coluna"      : col,
            "nulos"       : nulls,
            "completude %": round((1 - nulls / total) * 100, 1)
        })
    report = pd.DataFrame(rows)
    print(f"\n{'='*55}")
    print(f" Quality Report → {name}  ({total:,} registros)")
    print(f"{'='*55}")
    print(report.to_string(index=False))
    return report

# Relatório nas tabelas mais críticas
_ = quality_report("orders",      dfs["olist_orders_dataset"])
_ = quality_report("order_items", dfs["olist_order_items_dataset"])

In [ ]:
# ── 6. Tratamento básico antes de persistir ───────────────────────────────────

# orders: converter timestamps e remover duplicatas por order_id
orders = (
    dfs["olist_orders_dataset"]
    .withColumn("order_purchase_timestamp",
                F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_delivered_customer_date",
                F.to_timestamp("order_delivered_customer_date"))
    .dropDuplicates(["order_id"])
)

# order_items: garantir tipos numéricos
items = (
    dfs["olist_order_items_dataset"]
    .withColumn("price",         F.col("price").cast(DoubleType()))
    .withColumn("freight_value", F.col("freight_value").cast(DoubleType()))
)

# products: join com tradução de categorias
products = (
    dfs["olist_products_dataset"]
    .join(
        dfs["product_category_name_translation"],
        on="product_category_name",
        how="left"
    )
    .drop("product_category_name")
    .withColumnRenamed("product_category_name_english", "category")
)

print("Tratamentos aplicados ✅")
print(f"  orders   → {orders.count():,} registros")
print(f"  items    → {items.count():,} registros")
print(f"  products → {products.count():,} registros")

In [ ]:
# ── 7. Persistir como Parquet via PyArrow ─────────────────────────────────────
# Convertemos Spark DF → pandas DF → PyArrow Table → Parquet
# Isso demonstra a interoperabilidade entre as duas stacks.

def spark_to_parquet(spark_df, name: str):
    """Converte Spark DataFrame para Parquet usando PyArrow."""
    pdf   = spark_df.toPandas()
    table = pa.Table.from_pandas(pdf, preserve_index=False)
    path  = os.path.join(PARQUET_PATH, f"{name}.parquet")
    pq.write_table(
        table, path,
        compression="snappy",   # padrão em Data Lakes
        row_group_size=50_000
    )
    size_kb = os.path.getsize(path) / 1024
    print(f"  {name:<30} → {path}  ({size_kb:,.0f} KB, snappy)")

print("Escrevendo Parquet...\n")
spark_to_parquet(orders,   "orders")
spark_to_parquet(items,    "order_items")
spark_to_parquet(products, "products")
spark_to_parquet(dfs["olist_order_payments_dataset"],  "payments")
spark_to_parquet(dfs["olist_order_reviews_dataset"],   "reviews")
spark_to_parquet(dfs["olist_customers_dataset"],       "customers")
spark_to_parquet(dfs["olist_sellers_dataset"],         "sellers")
print("\nParquet escritos com sucesso ✅")

In [ ]:
# ── 8. Verificar schema dos Parquet gerados ───────────────────────────────────

for name in ["orders", "order_items", "products"]:
    path   = os.path.join(PARQUET_PATH, f"{name}.parquet")
    schema = pq.read_schema(path)
    print(f"\n── Schema: {name} ──")
    for field in schema:
        print(f"   {field.name:<45} {field.type}")

In [ ]:
# ── 9. Upload para Azure Blob Storage ─────────────────────────────────────────
# Simula camada Bronze de um Data Lake (ADLS Gen2 / Azure Blob).
# Configure a variável de ambiente AZURE_STORAGE_CONNECTION_STRING
# no painel do seu Storage Account → Access Keys.

def upload_to_azure(local_folder: str, container: str, prefix: str = "bronze/"):
    """Faz upload de todos os Parquet de uma pasta para o Azure Blob."""
    if not AZURE_CONN_STRING:
        print("⚠️  AZURE_STORAGE_CONNECTION_STRING não configurada.")
        print("   Pule esta célula se estiver rodando sem Azure.")
        return

    client = BlobServiceClient.from_connection_string(AZURE_CONN_STRING)

    # Cria container se não existir
    try:
        client.create_container(container)
        print(f"Container '{container}' criado.")
    except Exception:
        print(f"Container '{container}' já existe.")

    for filename in os.listdir(local_folder):
        if not filename.endswith(".parquet"):
            continue
        local_path = os.path.join(local_folder, filename)
        blob_path  = f"{prefix}{filename}"
        blob_client = client.get_blob_client(container=container, blob=blob_path)
        with open(local_path, "rb") as f:
            blob_client.upload_blob(f, overwrite=True)
        print(f"  ✅ Uploaded: {blob_path}")

upload_to_azure(PARQUET_PATH, AZURE_CONTAINER)
print("\nPipeline de ingestão finalizado 🚀")

## ✅ Resumo — Notebook 01

| Etapa | Ferramenta | Resultado |
|-------|------------|----------|
| Leitura CSV | PySpark | 8 DataFrames carregados |
| Validação de qualidade | PySpark + pandas | Relatório de nulos por coluna |
| Tratamento básico | PySpark | Tipos corrigidos, duplicatas removidas |
| Persistência | PyArrow (Snappy Parquet) | 7 arquivos .parquet gerados |
| Armazenamento em nuvem | Azure Blob Storage | Camada Bronze no container `olist-datalake` |

**Próximo passo →** `02_transformacao.ipynb` — joins, cálculo de CPV, DRE por categoria com DuckDB + pandas + numpy.